# 00. はじめに：実行環境の確認（最初にこれを実行してください）

本講座「生成AIによる画像復元技術と最新AIモデルの応用」の演習ノートブックです。

## 使い方（Google Colab）
1. **［ファイル］→［ドライブにコピーを保存］** で自分用のコピーを作る（元のノートブックは編集できません）
2. **［ランタイム］→［ランタイムのタイプを変更］→「T4 GPU」** を選んで保存
3. 上のセルから順に ▶ ボタン（または Shift+Enter）で実行
4. 終わったら **［ランタイム］→［ランタイムを接続解除して削除］**

## 使い方（ローカルの GPU PC）
`docs/ローカル環境構築ガイド.pdf` の手順で Python 環境を作り、Jupyter Lab / VS Code で開いて実行します。コードは Colab と共通です。

## ノートブック一覧
| No. | 内容 | 推奨環境 |
|---|---|---|
| 01 | 劣化モデルと古典的復元（Wiener・補間） | CPU可 |
| 02 | GAN・拡散モデル・CLIP の仕組み | T4 |
| 03 | 超解像：Real-ESRGAN vs 拡散モデル | T4 |
| 04 | Inpainting / Outpainting・境界のなじませ | T4 |
| 05 | Deblurring / Denoising（DPIR・SCUNet） | T4 |
| 06 | 条件付け：img2img・ControlNet・IP-Adapter | T4 |
| 07 | 外観検査・異常検知・疑似欠陥生成 | T4 |
| 08 | 評価指標とハルシネーション対策 | T4 |
| 09 | 軽量化：蒸留・量子化・ONNX | T4 |
| 10 | LoRA による現場特化チューニング | L4推奨（有料） |
| 11 | マルチモーダル：言葉で指示する復元・VLM | L4推奨（有料） |
| 12 | 物理モデルを組み込んだ AI：スマホ→一眼・塑性変形 | T4 |
| 13 | 奥行きで変わるピンボケ：場所ごとの逆カーネル | T4 |
| 14 | VLM による外観検査：違和感を言葉で報告 | T4 |
| 15 | ComfyUI on Colab：SUPIR による高品質復元 | A100推奨（有料・ハイメモリ） |
| 16 | 超解像・顔復元の手法比較（SUPIR・DiffBIR・CodeFormer など） | A100推奨（有料・ハイメモリ） |

In [1]:
# ===== 共通セットアップ（このセルを最初に1回だけ実行してください） =====
import os, sys, subprocess, time, urllib.request

IN_COLAB = "google.colab" in sys.modules

def pip_install(*pkgs):
    """足りないライブラリを静かにインストールする（Colab用）"""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

if IN_COLAB:
    # Colabには torch / transformers / diffusers などが最初から入っています。
    # 不足分だけ追加します（1〜2分）。
    pip_install("spandrel", "lpips", "diffusers>=0.30", "accelerate", "peft")

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"実行環境: {'Google Colab' if IN_COLAB else 'ローカルPC'} / torch {torch.__version__}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  VRAM: {p.total_memory / 1024**3:.1f} GB")
else:
    print("GPUが見つかりません。Colabでは［ランタイム］→［ランタイムのタイプを変更］→ T4 GPU を選んでください。")

WEIGHTS_DIR = os.path.abspath("weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

def download(url, name=None):
    """学習済み重みをダウンロード（2回目以降はキャッシュを使う）"""
    path = os.path.join(WEIGHTS_DIR, name or url.split("/")[-1])
    if not os.path.exists(path):
        print("downloading", url)
        urllib.request.urlretrieve(url, path + ".part")      # 途中で止めても壊れたファイルが残らないように
        os.replace(path + ".part", path)
    return path

def load_sample(name="astronaut", size=None):
    """scikit-image同梱のサンプル画像（ネット不要）。name: astronaut / coffee / chelsea / rocket"""
    from skimage import data
    img = Image.fromarray(getattr(data, name)()).convert("RGB")
    if size:
        img = img.resize((size, size) if isinstance(size, int) else size, Image.BICUBIC)
    return img

def load_portrait():
    """顔が大きく写った 512x512 の高画質画像（Kodak 標準テスト画像 kodim04 の上部を切り出し）。
    Kodak 画像は画像処理の評価用に広く使われているフリーのテスト画像（r0k.us/graphics/kodak/）。
    ダウンロードできない場合は astronaut で代用。"""
    try:
        img = Image.open(download("https://r0k.us/graphics/kodak/kodak/kodim04.png")).convert("RGB")
        return img.crop((0, 160, 512, 672))
    except Exception as e:
        print("kodim04 を取得できなかったため astronaut を使います:", e)
        return load_sample("astronaut")

def to_np(img):
    """PIL → float32 [0,1] の (H,W,3) 配列"""
    return np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0

def to_pil(arr):
    return Image.fromarray((np.clip(arr, 0, 1) * 255 + 0.5).astype(np.uint8))

def to_tensor(img):
    """PIL / ndarray → (1,3,H,W) テンソル"""
    a = to_np(img) if isinstance(img, Image.Image) else img
    return torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32)).permute(2, 0, 1)[None].contiguous()

def tensor_to_pil(t):
    return to_pil(t[0].detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy())

def psnr(a, b):
    a, b = to_np(a) if isinstance(a, Image.Image) else a, to_np(b) if isinstance(b, Image.Image) else b
    return 10 * np.log10(1.0 / np.mean((a - b) ** 2))

def ssim(a, b):
    from skimage.metrics import structural_similarity
    a, b = to_np(a) if isinstance(a, Image.Image) else a, to_np(b) if isinstance(b, Image.Image) else b
    return structural_similarity(a, b, channel_axis=2, data_range=1.0)

def show(images, titles=None, size=4, rows=1):
    """画像を横に並べて表示（タイトルは文字化け防止のため英数字推奨）"""
    n = len(images)
    cols = (n + rows - 1) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            im = images[i]
            if isinstance(im, np.ndarray) and im.ndim == 2:
                ax.imshow(im, cmap="gray")                      # 2次元配列（マップ・スペクトル）は自動スケール
            else:
                if isinstance(im, np.ndarray) and im.dtype != np.uint8:
                    im = np.clip(im, 0, 1)                      # float 画像は 0〜1 に収める
                ax.imshow(im)
            if titles:
                ax.set_title(titles[i], fontsize=11)
    plt.tight_layout()
    plt.show()

def gpu_mem():
    if DEVICE == "cuda":
        print(f"GPUメモリ使用量(ピーク): {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

実行環境: ローカルPC / torch 2.11.0+cu128
GPU: NVIDIA GeForce RTX 5090 Laptop GPU  VRAM: 23.9 GB


## 実行環境の詳細チェック

In [2]:
import shutil, platform
print("Python  :", platform.python_version())
try:
    import psutil
    print(f"システムRAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
except ImportError:
    pass
print(f"ディスク空き: {shutil.disk_usage('.').free / 1024**3:.0f} GB")

if DEVICE == "cuda":
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\nGPU メモリ {vram:.1f} GB で実行できるノートブック:")
    print("  01〜09: OK" if vram >= 7 else "  01〜05, 09 程度（他はメモリ不足の可能性）")
    print("  10〜14: OK" if vram >= 14 else "  10〜14: メモリ不足の可能性（L4以上を推奨）")
    print("  15・16: OK（システムRAMも 40GB 以上必要）" if vram >= 22 else "  15・16: GPU メモリ不足（A100 + ハイメモリを推奨）")
else:
    print("\n⚠ GPU なし：ノートブック 01 以外は非常に遅くなります。")

Python  : 3.12.11
システムRAM: 63.4 GB
ディスク空き: 1445 GB

GPU メモリ 23.9 GB で実行できるノートブック:
  01〜09: OK
  10〜14: OK
  15・16: OK（システムRAMも 40GB 以上必要）


## 簡単な動作テスト：GPU で行列計算

In [3]:
a = torch.randn(4096, 4096, device=DEVICE, dtype=DTYPE)
t = time.time()
for _ in range(20):
    b = a @ a
if DEVICE == "cuda":
    torch.cuda.synchronize()
dt = time.time() - t
print(f"4096x4096 行列積 x20: {dt:.2f}s  （約 {20 * 2 * 4096**3 / dt / 1e12:.1f} TFLOPS）")

4096x4096 行列積 x20: 0.13s  （約 21.1 TFLOPS）


## 結果を Google Drive に保存する方法（Colab）
ランタイムが切断されると、作業フォルダのファイルはすべて消えます。残したい結果は Drive に保存してください。

In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # 初回は許可を求める画面が出ます
    os.makedirs("/content/drive/MyDrive/genai_restoration", exist_ok=True)
    load_sample("astronaut").save("/content/drive/MyDrive/genai_restoration/test.png")
    print("Drive に保存しました: MyDrive/genai_restoration/test.png")
else:
    print("ローカル実行なので Drive マウントは不要です。")

ローカル実行なので Drive マウントは不要です。


## Colab の有料枠について（要点）
- 有料枠は **コンピューティングユニット（CU）のプリペイド制**。GPU ランタイムに **接続している間** 減り続けます（計算していなくても）
- T4 は約 1〜2 CU/時、A100 は約 5〜15 CU/時（変動します。右上のリソース表示で確認）
- **使い終わったら［ランタイムを接続解除して削除］** ／ 月額プランは自動更新なので不要になったら解約
- 詳しくは `docs/Colab有料枠ガイド.pdf`